# 🤟 Qyran Sign Language Model - Holistic (255 features)
## Transformer + BiLSTM | Incremental Learning | Target: 90%+

**Шаги:**
1. Загрузка данных с Google Drive
2. Выбор 10-20 жестов для начала
3. Data Augmentation
4. Transformer модель
5. Тренировка
6. Экспорт в TensorFlow.js

In [ ]:
# 1. Подключаем Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Распаковка архива
import os
import zipfile

zip_path = '/content/drive/MyDrive/landmarks_holistic.zip'
extract_dir = '/content/landmarks_holistic'

if not os.path.exists(extract_dir):
    print('Распаковываем...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content/')
    print('Готово!')
else:
    print('Уже распаковано')

# Проверяем
train_count = len(os.listdir(os.path.join(extract_dir, 'train')))
test_count = len(os.listdir(os.path.join(extract_dir, 'test')))
print(f'Train: {train_count}, Test: {test_count}')

In [ ]:
# 3. Загружаем label map и annotations
import json
import pandas as pd
import numpy as np

with open(os.path.join(extract_dir, 'label_map.json'), 'r') as f:
    full_label_map = json.load(f)

with open(os.path.join(extract_dir, 'config.json'), 'r') as f:
    config = json.load(f)

print(f"Всего классов: {len(full_label_map)}")
print(f"Features per frame: {config['features_per_frame']}")
print(f"Sequence length: {config['sequence_length']}")
print(f"Feature layout: {config['feature_layout']}")

In [ ]:
# 4. Загрузка annotations.csv с Google Drive
# Скопируй annotations.csv из датасета на Google Drive тоже!
# Или загрузи напрямую:

ann_path = '/content/drive/MyDrive/annotations.csv'
if not os.path.exists(ann_path):
    # Попробуем найти в другом месте
    ann_path = '/content/drive/MyDrive/slovo/annotations.csv'

df = pd.read_csv(ann_path, sep='\t')
print(f'Всего записей: {len(df)}')
print(df.head())

In [ ]:
# 5. Выбираем TOP-N самых частых жестов для начала
# Начинаем с 20 жестов для высокой точности

NUM_CLASSES = 20  # Меняй: 10, 20, 50, 100...

# Берём самые частые жесты
top_gestures = df['text'].value_counts().head(NUM_CLASSES).index.tolist()
print(f'Выбрано {NUM_CLASSES} жестов:')
for i, g in enumerate(top_gestures):
    count = df[df['text'] == g].shape[0]
    print(f'  {i+1}. {g} ({count} видео)')

# Создаём новый label map только для выбранных жестов
selected_label_map = {gesture: idx for idx, gesture in enumerate(top_gestures)}
print(f'\nLabel map: {selected_label_map}')

In [ ]:
# 6. Загрузка данных
SEQUENCE_LENGTH = config['sequence_length']  # 60
FEATURES = config['features_per_frame']  # 255

def load_data(split, selected_gestures, label_map):
    is_train = (split == 'train')
    split_df = df[df['train'] == is_train]
    # Фильтруем только выбранные жесты
    split_df = split_df[split_df['text'].isin(selected_gestures)]

    X, y = [], []
    missing = 0
    for _, row in split_df.iterrows():
        vid = row['attachment_id']
        npy_path = os.path.join(extract_dir, split, f'{vid}.npy')
        if os.path.exists(npy_path):
            data = np.load(npy_path)
            X.append(data)
            y.append(label_map[row['text']])
        else:
            missing += 1

    print(f'{split}: loaded {len(X)} samples, missing {missing}')
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)

X_train, y_train = load_data('train', top_gestures, selected_label_map)
X_test, y_test = load_data('test', top_gestures, selected_label_map)

print(f'\nX_train: {X_train.shape}')  # (N, 60, 255)
print(f'y_train: {y_train.shape}')
print(f'X_test: {X_test.shape}')
print(f'y_test: {y_test.shape}')

In [ ]:
# 7. Data Augmentation
def augment_sequence(seq):
    """Аугментация одной последовательности"""
    augmented = seq.copy()

    # 1. Шум (гауссовский)
    if np.random.random() < 0.5:
        noise = np.random.normal(0, 0.005, augmented.shape)
        augmented = augmented + noise

    # 2. Масштабирование
    if np.random.random() < 0.5:
        scale = np.random.uniform(0.9, 1.1)
        augmented = augmented * scale

    # 3. Временной сдвиг (случайный сдвиг фреймов)
    if np.random.random() < 0.3:
        shift = np.random.randint(-3, 4)
        augmented = np.roll(augmented, shift, axis=0)

    # 4. Изменение скорости (ресемплинг)
    if np.random.random() < 0.3:
        speed = np.random.uniform(0.8, 1.2)
        indices = np.linspace(0, len(augmented)-1, int(len(augmented)*speed))
        indices = np.clip(indices, 0, len(augmented)-1).astype(int)
        stretched = augmented[indices]
        # Вернуть к исходной длине
        target_indices = np.linspace(0, len(stretched)-1, len(augmented)).astype(int)
        augmented = stretched[target_indices]

    return augmented.astype(np.float32)

# Аугментируем train данные x3
AUGMENT_FACTOR = 3
X_aug = [X_train]
y_aug = [y_train]

for i in range(AUGMENT_FACTOR):
    X_new = np.array([augment_sequence(x) for x in X_train])
    X_aug.append(X_new)
    y_aug.append(y_train.copy())
    print(f'Augmentation round {i+1} done')

X_train_aug = np.concatenate(X_aug, axis=0)
y_train_aug = np.concatenate(y_aug, axis=0)

# Shuffle
perm = np.random.permutation(len(X_train_aug))
X_train_aug = X_train_aug[perm]
y_train_aug = y_train_aug[perm]

print(f'\nПосле аугментации:')
print(f'X_train: {X_train_aug.shape}')
print(f'y_train: {y_train_aug.shape}')

In [ ]:
# 8. Нормализация
from sklearn.preprocessing import StandardScaler

# Reshape для scaler
scaler = StandardScaler()
X_train_flat = X_train_aug.reshape(-1, FEATURES)
scaler.fit(X_train_flat)

X_train_norm = scaler.transform(X_train_flat).reshape(X_train_aug.shape)
X_test_norm = scaler.transform(X_test.reshape(-1, FEATURES)).reshape(X_test.shape)

print('Нормализация готова')
print(f'Mean: {X_train_norm.mean():.4f}, Std: {X_train_norm.std():.4f}')

In [ ]:
# 9. Модель: Transformer + BiLSTM
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation='gelu'),
            layers.Dense(embed_dim),
        ])
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(rate)
        self.drop2 = layers.Dropout(rate)

    def call(self, x, training=False):
        attn = self.att(x, x, training=training)
        attn = self.drop1(attn, training=training)
        x = self.norm1(x + attn)
        ffn_out = self.ffn(x)
        ffn_out = self.drop2(ffn_out, training=training)
        return self.norm2(x + ffn_out)

def build_model(seq_len, features, num_classes):
    inputs = layers.Input(shape=(seq_len, features))

    # Projection
    x = layers.Dense(128)(inputs)

    # Positional encoding (learnable)
    positions = tf.range(start=0, limit=seq_len, delta=1)
    pos_embedding = layers.Embedding(input_dim=seq_len, output_dim=128)(positions)
    x = x + pos_embedding

    # Transformer blocks
    x = TransformerBlock(128, 8, 256, 0.2)(x)
    x = TransformerBlock(128, 8, 256, 0.2)(x)

    # BiLSTM
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Bidirectional(layers.LSTM(64))(x)
    x = layers.Dropout(0.3)(x)

    # Classification head
    x = layers.Dense(128, activation='gelu')(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(64, activation='gelu')(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs)
    return model

model = build_model(SEQUENCE_LENGTH, FEATURES, NUM_CLASSES)
model.summary()

In [ ]:
# 10. Компиляция и тренировка
optimizer = keras.optimizers.Adam(learning_rate=1e-3)

model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy', factor=0.5, patience=5, min_lr=1e-6, verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=15, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        '/content/best_model.keras', monitor='val_accuracy',
        save_best_only=True, verbose=1
    )
]

history = model.fit(
    X_train_norm, y_train_aug,
    validation_data=(X_test_norm, y_test),
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# 11. Результаты
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'], label='Train')
ax1.plot(history.history['val_accuracy'], label='Test')
ax1.set_title('Accuracy')
ax1.legend()
ax1.grid(True)

ax2.plot(history.history['loss'], label='Train')
ax2.plot(history.history['val_loss'], label='Test')
ax2.set_title('Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

# Финальная точность
test_loss, test_acc = model.evaluate(X_test_norm, y_test)
print(f'\n🎯 Test Accuracy: {test_acc*100:.1f}%')

In [ ]:
# 12. Confusion Matrix
from sklearn.metrics import confusion_matrix, classification_report

y_pred = model.predict(X_test_norm).argmax(axis=1)

# Classification report
inv_label_map = {v: k for k, v in selected_label_map.items()}
target_names = [inv_label_map[i] for i in range(NUM_CLASSES)]

print(classification_report(y_test, y_pred, target_names=target_names))

In [ ]:
# 13. Экспорт в TensorFlow.js
!pip install tensorflowjs -q

import tensorflowjs as tfjs

# Загружаем лучшую модель
best_model = keras.models.load_model('/content/best_model.keras')

# Экспорт
export_dir = '/content/tfjs_model'
tfjs.converters.save_keras_model(best_model, export_dir)

print(f'\n✅ Модель экспортирована в {export_dir}')
print('Файлы:')
for f in os.listdir(export_dir):
    size = os.path.getsize(os.path.join(export_dir, f))
    print(f'  {f} ({size/1024:.1f} KB)')

In [ ]:
# 14. Сохраняем всё на Google Drive
import shutil

save_dir = '/content/drive/MyDrive/qyran_model'
os.makedirs(save_dir, exist_ok=True)

# Копируем TF.js модель
tfjs_save = os.path.join(save_dir, 'tfjs_model')
if os.path.exists(tfjs_save):
    shutil.rmtree(tfjs_save)
shutil.copytree(export_dir, tfjs_save)

# Сохраняем label map и scaler
with open(os.path.join(save_dir, 'label_map.json'), 'w') as f:
    json.dump(selected_label_map, f, ensure_ascii=False, indent=2)

# Сохраняем scaler параметры для JS
scaler_params = {
    'mean': scaler.mean_.tolist(),
    'scale': scaler.scale_.tolist()
}
with open(os.path.join(save_dir, 'scaler.json'), 'w') as f:
    json.dump(scaler_params, f)

# Сохраняем config
model_config = {
    'sequence_length': SEQUENCE_LENGTH,
    'features_per_frame': FEATURES,
    'num_classes': NUM_CLASSES,
    'gesture_names': top_gestures,
    'accuracy': float(test_acc)
}
with open(os.path.join(save_dir, 'model_config.json'), 'w') as f:
    json.dump(model_config, f, ensure_ascii=False, indent=2)

print(f'✅ Всё сохранено в {save_dir}')
print('\nФайлы:')
for root, dirs, files in os.walk(save_dir):
    for f in files:
        fp = os.path.join(root, f)
        print(f'  {os.path.relpath(fp, save_dir)} ({os.path.getsize(fp)/1024:.1f} KB)')

In [ ]:
# 15. Скачать TF.js модель
# Запакуем для скачивания
shutil.make_archive('/content/qyran_tfjs_model', 'zip', save_dir)
print('📥 Скачай файл: /content/qyran_tfjs_model.zip')

from google.colab import files
files.download('/content/qyran_tfjs_model.zip')